# OCR Library Compatibility Checker

Run this notebook to find which OCR library works on your system.

Tests: PaddleOCR, EasyOCR, Tesseract

In [ ]:
# Step 1: Check Python version
import sys
import platform

print("=" * 60)
print("SYSTEM INFO")
print("=" * 60)
print(f"Python version: {sys.version}")
print(f"Platform: {platform.system()} {platform.release()}")
print(f"Architecture: {platform.machine()}")

py_version = sys.version_info
print(f"\nPython {py_version.major}.{py_version.minor}.{py_version.micro}")

if py_version >= (3, 12):
    print("WARNING: Python 3.12+ may have compatibility issues with some OCR libraries")
elif py_version >= (3, 10):
    print("OK: Python 3.10-3.11 has best compatibility")

In [ ]:
# Step 2: Check what's already installed
import subprocess

def check_package(name):
    try:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "show", name],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            for line in result.stdout.split('\n'):
                if line.startswith('Version:'):
                    return line.split(':')[1].strip()
        return None
    except:
        return None

packages = [
    'paddlepaddle', 'paddleocr', 'paddlex',
    'easyocr', 'pytesseract',
    'langchain', 'langchain-community', 'langchain-core',
    'torch', 'opencv-python', 'numpy', 'pillow'
]

print("=" * 60)
print("INSTALLED PACKAGES")
print("=" * 60)
for pkg in packages:
    ver = check_package(pkg)
    status = ver if ver else "NOT INSTALLED"
    print(f"{pkg:25s} : {status}")

---
## Test 1: EasyOCR

In [ ]:
# Install EasyOCR
!pip install easyocr -q

In [ ]:
# Test EasyOCR import
print("Testing EasyOCR...")
try:
    import easyocr
    print(f"  Version: {easyocr.__version__}")
    
    # Test initialization (downloads models)
    reader = easyocr.Reader(['en'], gpu=False, verbose=False)
    print("  Import: OK")
    print("  Init: OK")
    print("  EASYOCR: WORKS!")
    EASYOCR_WORKS = True
except Exception as e:
    print(f"  FAILED: {e}")
    EASYOCR_WORKS = False

---
## Test 2: PaddleOCR (Method A - Latest with langchain fix)

In [ ]:
# Install PaddleOCR with compatible langchain
!pip uninstall paddlepaddle paddleocr paddlex langchain langchain-community langchain-core -y -q
!pip install paddlepaddle -q
!pip install paddleocr -q
!pip install langchain==0.1.0 langchain-community==0.0.13 -q

In [ ]:
# Test PaddleOCR import (Method A)
print("Testing PaddleOCR (Method A: Latest + langchain 0.1.0)...")
try:
    import paddle
    print(f"  PaddlePaddle: {paddle.__version__}")
    
    from paddleocr import PaddleOCR
    print("  Import: OK")
    
    ocr = PaddleOCR(use_doc_orientation_classify=False, use_doc_unwarping=False, use_textline_orientation=False)
    print("  Init: OK")
    print("  PADDLEOCR METHOD A: WORKS!")
    PADDLE_A_WORKS = True
except Exception as e:
    print(f"  FAILED: {e}")
    PADDLE_A_WORKS = False

---
## Test 3: PaddleOCR (Method B - Older stable version)

In [ ]:
# Only run if Method A failed
if not PADDLE_A_WORKS:
    print("Method A failed, trying Method B...")
    !pip uninstall paddlepaddle paddleocr paddlex -y -q
    !pip install paddlepaddle==2.6.2 -q
    !pip install paddleocr==2.8.1 -q
else:
    print("Method A worked, skipping Method B")

In [ ]:
# Test PaddleOCR import (Method B)
PADDLE_B_WORKS = False

if not PADDLE_A_WORKS:
    print("Testing PaddleOCR (Method B: paddlepaddle 2.6.2 + paddleocr 2.8.1)...")
    try:
        import importlib
        import paddle
        importlib.reload(paddle)
        print(f"  PaddlePaddle: {paddle.__version__}")
        
        from paddleocr import PaddleOCR
        print("  Import: OK")
        
        ocr = PaddleOCR(use_angle_cls=True, lang='en')
        print("  Init: OK")
        print("  PADDLEOCR METHOD B: WORKS!")
        PADDLE_B_WORKS = True
    except Exception as e:
        print(f"  FAILED: {e}")
        PADDLE_B_WORKS = False

---
## Test 4: Tesseract

In [ ]:
# Install Tesseract (Colab)
import platform
if 'google.colab' in str(get_ipython()) or platform.system() == 'Linux':
    !apt-get install -y tesseract-ocr tesseract-ocr-ara tesseract-ocr-fra -qq
    !pip install pytesseract -q
    print("Tesseract installed (Linux/Colab)")
else:
    print("For Windows: Install Tesseract from https://github.com/UB-Mannheim/tesseract/wiki")
    !pip install pytesseract -q

In [ ]:
# Test Tesseract
print("Testing Tesseract...")
try:
    import pytesseract
    
    # Check if tesseract binary exists
    version = pytesseract.get_tesseract_version()
    print(f"  Version: {version}")
    
    langs = pytesseract.get_languages()
    print(f"  Languages: {', '.join(langs[:10])}...")
    
    print("  TESSERACT: WORKS!")
    TESSERACT_WORKS = True
except Exception as e:
    print(f"  FAILED: {e}")
    TESSERACT_WORKS = False

---
## FINAL RESULTS

In [ ]:
print("\n" + "=" * 60)
print("COMPATIBILITY RESULTS")
print("=" * 60)

results = []

# EasyOCR
if 'EASYOCR_WORKS' in dir() and EASYOCR_WORKS:
    print("EasyOCR:              WORKS")
    results.append(('EasyOCR', True))
else:
    print("EasyOCR:              FAILED")
    results.append(('EasyOCR', False))

# PaddleOCR Method A
if 'PADDLE_A_WORKS' in dir() and PADDLE_A_WORKS:
    print("PaddleOCR (latest):   WORKS")
    results.append(('PaddleOCR_latest', True))
else:
    print("PaddleOCR (latest):   FAILED")
    results.append(('PaddleOCR_latest', False))

# PaddleOCR Method B
if 'PADDLE_B_WORKS' in dir() and PADDLE_B_WORKS:
    print("PaddleOCR (2.8.1):    WORKS")
    results.append(('PaddleOCR_2.8.1', True))
else:
    print("PaddleOCR (2.8.1):    FAILED")
    results.append(('PaddleOCR_2.8.1', False))

# Tesseract
if 'TESSERACT_WORKS' in dir() and TESSERACT_WORKS:
    print("Tesseract:            WORKS")
    results.append(('Tesseract', True))
else:
    print("Tesseract:            FAILED")
    results.append(('Tesseract', False))

print("\n" + "=" * 60)
print("RECOMMENDATION")
print("=" * 60)

working = [name for name, status in results if status]

if 'PaddleOCR_latest' in working or 'PaddleOCR_2.8.1' in working:
    print("USE: PaddleOCR (highest accuracy for ID cards)")
elif 'Tesseract' in working:
    print("USE: Tesseract + MRZ validation (good accuracy with tuning)")
elif 'EasyOCR' in working:
    print("USE: EasyOCR (easier setup, moderate accuracy)")
else:
    print("No OCR library worked. Check errors above.")

print(f"\nWorking libraries: {working}")

---
## Quick Test with Image (if any library works)

In [ ]:
# Upload test image
try:
    from google.colab import files
    print("Upload a test image:")
    uploaded = files.upload()
    TEST_IMAGE = list(uploaded.keys())[0]
    print(f"Using: {TEST_IMAGE}")
except:
    TEST_IMAGE = "front_cropped.jpg"  # Local fallback
    print(f"Using local file: {TEST_IMAGE}")

In [ ]:
# Quick OCR test with the first working library
import cv2
import os

if not os.path.exists(TEST_IMAGE):
    print(f"Image not found: {TEST_IMAGE}")
else:
    img = cv2.imread(TEST_IMAGE)
    print(f"Image loaded: {img.shape[1]}x{img.shape[0]}")
    
    if 'PADDLE_A_WORKS' in dir() and PADDLE_A_WORKS:
        print("\nTesting with PaddleOCR (latest)...")
        result = ocr.predict(input=TEST_IMAGE)
        for res in result:
            res.print()
    
    elif 'PADDLE_B_WORKS' in dir() and PADDLE_B_WORKS:
        print("\nTesting with PaddleOCR (2.8.1)...")
        result = ocr.ocr(TEST_IMAGE, cls=True)
        for line in result[0]:
            print(line[1][0])  # text
    
    elif 'EASYOCR_WORKS' in dir() and EASYOCR_WORKS:
        print("\nTesting with EasyOCR...")
        result = reader.readtext(TEST_IMAGE)
        for detection in result:
            print(detection[1])  # text
    
    elif 'TESSERACT_WORKS' in dir() and TESSERACT_WORKS:
        print("\nTesting with Tesseract...")
        from PIL import Image
        text = pytesseract.image_to_string(Image.open(TEST_IMAGE))
        print(text)